In [ ]:
import skimage
from skimage import filters
import numpy as np
import matplotlib.pyplot as plt
import hyperspy.api as hs
import sys
sys.path.append('..')
import roi_tools

s = hs.load('../data/images/Jaume LFO/HAADF_Buena.dm3')
left_bound = 20
right_bound = 2048-20
start_pixel = 200
end_pixel = 2048-100

# s = hs.load('../data/images/Jaume LFO EELS/EEL SI 22 13/Integrated Shifted Spectrum Image adf.dm3')
# left_bound = 0 # TUNE THIS
# right_bound = 185 # TUNE THIS
# start_pixel = 20 # TUNE THIS
# end_pixel = 235 # TUNE THIS

roi = roi_tools.ROI(s, left_bound, right_bound, start_pixel, end_pixel)
roi.build_grid_dict()
roi.get_atom_types()

fe_mask = np.zeros_like(roi.grid, dtype=bool)
for (i, j), patch in np.ndenumerate(roi.grid):
    if patch is not None and getattr(patch, 'atom_type', None) == 'Fe':
        fe_mask[i, j] = True

In [ ]:
# Fe
max14 = 1.030461
max8 = 1.029236
max4 = 1.025412

# max14 = 1.029597
# max8 = 1.035865
# max4 = 1.025412

# Lu
# max14 = 1.032341
# max8 = 1.029938
# max4 = 1.029670

metric = 'mean_intensity'

# 14
dis = np.array([0, 0, -1, -1, -1, 1, 1, 1, -2, -2, -2, 2, 2, 2])
djs = np.array([-1, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1])
# dis = np.array([0, 0, -1, -1, -1, 1, 1, 1, -2, 2])
# djs = np.array([-1, -1, 0, 1, -1, 0, 1, -1, 0, 0])

djs = djs * 2
roi.set_vicinity_coords(dis, djs)
roi.get_vicinity(metric=metric)
roi.get_relative_vicinity(metric=metric)

raw_higher_map_14 = roi.relative_vicinity.astype(float) > max14
higher_map_14 = raw_higher_map_14 & fe_mask
relative_vicinity_14 = roi.relative_vicinity.astype(float)

# 8
dis = np.array([0, 0, -1, -1, -1, 1, 1, 1])
djs = np.array([-1, 1, -1, 0, 1, -1, 0, 1])
# dis = np.array([-1, 1,-2, 2])
# djs = np.array([0, 0, 0, 0])
djs = djs * 2
roi.set_vicinity_coords(dis, djs)
roi.get_vicinity(metric=metric)
roi.get_relative_vicinity(metric=metric)

raw_higher_map_8 = roi.relative_vicinity.astype(float) > max8
higher_map_8 = raw_higher_map_8 & fe_mask
relative_vicinity_8 = roi.relative_vicinity.astype(float)

# 4
dis = np.array([0, 0, -1, 1])
djs = np.array([-1, 1, 0, 0])
djs = djs * 2
roi.set_vicinity_coords(dis, djs)
roi.get_vicinity(metric=metric)
roi.get_relative_vicinity(metric=metric)

raw_higher_map_4 = roi.relative_vicinity.astype(float) > max4
higher_map_4 = raw_higher_map_4 & fe_mask
relative_vicinity_4 = roi.relative_vicinity.astype(float)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

y_ax_max = 1.1
y_ax_min = 0.9


def plot_vicinity_comparison(roi, fe_mask, configs):
    """
    Generates a three-panel horizontal histogram plot where subplots are adjacent.
    Bars above the simulation cutoff are highlighted in a different color.
    """
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial'],
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'axes.labelsize': 18,     
        'xtick.labelsize': 14,    
        'ytick.labelsize': 14,
        'text.usetex': False 
    })

    fig, axes = plt.subplots(1, 3, figsize=(14, 6), sharey=True)
    
    color_pristine_hist = '#1f77b4'
    color_defect_hist = '#D6204E' 
    color_pristine_stat = "#001cbc"
    color_defect_stat = "#9D0229"
    bin_width = 0.0015
    bins = np.arange(y_ax_min, y_ax_max + bin_width, bin_width)

    for ax, (label, data_array, cutoff) in zip(axes, configs):
        data = data_array[fe_mask].astype(float)
        data = data[~np.isnan(data)]
        mean_val = np.mean(data)
        
        # Calculate histogram manually to control individual bar colors
        counts, edges = np.histogram(data, bins=bins)
        centers = (edges[:-1] + edges[1:]) / 2
        
        for count, center in zip(counts, centers):
            bar_color = color_defect_hist if center > cutoff else color_pristine_hist
            ax.barh(center, count, height=bin_width, color=bar_color, alpha=0.7, zorder=3)
        
        ax.axhline(y=mean_val, color=color_pristine_stat, linestyle='--', linewidth=1.2, zorder=4)
        ax.axhline(y=cutoff, color=color_pristine_stat, linestyle='-', linewidth=2, zorder=4) 

        trans = ax.get_yaxis_transform()
        ax.text(0.98, cutoff + 0.0005, f'{cutoff:.4f}', color=color_pristine_stat, transform=trans, 
                va='bottom', ha='right', fontsize=14, fontweight='bold', zorder=5)
        ax.text(0.98, mean_val + 0.0005, f'{mean_val:.4f}', color=color_pristine_stat, transform=trans, 
                va='bottom', ha='right', fontsize=14, zorder=5)

        ax.set_title(f'{label} Atoms', fontsize=18, pad=10) 
        ax.tick_params(direction='in', top=False, right=False, length=6)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4, integer=True, prune='upper'))
        ax.set_xlim(0, 32)

    # Proxy artists for legend since we plotted bars individually
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=color_defect_hist, alpha=0.7, label='Potential Antisites'),
        Patch(facecolor=color_pristine_hist, alpha=0.7, label='Potential Non-Antisites'),
        Line2D([0], [0], color=color_pristine_stat, linestyle='-', label='Simulation Cutoff', linewidth=2),
        Line2D([0], [0], color=color_pristine_stat, linestyle='--', label='Mean')
    ]

    axes[1].set_xlabel('Counts')
    axes[0].set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
    axes[0].set_ylim(y_ax_min, y_ax_max)

    axes[-1].legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1.0), 
                    frameon=False, fontsize=15)

    fig.subplots_adjust(wspace=0.0, right=0.78, left=0.08, bottom=0.12, top=0.88) 
    plt.show()

vicinity_configs = [
    ('4', relative_vicinity_4, max4),
    ('8', relative_vicinity_8, max8),
    ('14', relative_vicinity_14, max14)
]

plot_vicinity_comparison(roi, fe_mask, vicinity_configs)

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

"""
Overlays tiered detection boxes onto the ROI based on how many metrics 
(Vicinity 4, 8, or 14) flagged a potential defect.
"""

roi_plotter = roi_tools.ROIPlotter(roi, title=None)

# Calculate detection counts across all three metrics
mask_sum = higher_map_4.astype(int) + higher_map_8.astype(int) + higher_map_14.astype(int)

# Plot Tier 1: Detected by exactly 1 metric (High transparency)
roi_plotter.add_boolean_patches_overlay(mask_sum == 1, color="#FFCFD9", alpha=0.6)

# Plot Tier 2: Detected by exactly 2 metrics (Medium transparency)
roi_plotter.add_boolean_patches_overlay(mask_sum == 2, color="#BB5D75", alpha=0.6)

# Plot Tier 3: Detected by all 3 metrics (No alpha/Opaque)
roi_plotter.add_boolean_patches_overlay(mask_sum == 3, color="#820021", alpha=0.6)

# Create tiered legend proxies to explain the alpha levels
tier1_proxy = mpatches.Patch(color='#FFCFD9', alpha=0.6, label='1 Metric', edgecolor='none')
tier2_proxy = mpatches.Patch(color='#BB5D75', alpha=0.6, label='2 Metrics', edgecolor='none')
tier3_proxy = mpatches.Patch(color='#820021', alpha=0.6, label='3 Metrics', edgecolor='none')

roi_plotter.ax.legend(
    handles=[tier1_proxy, tier2_proxy, tier3_proxy], 
    loc='upper left', 
    bbox_to_anchor=(1.01, 1.0), 
    frameon=False, 
    fontsize=15,
    title="Metrics Met",
    title_fontsize=15,
    handletextpad=0.5,
    borderaxespad=0
)

# Adjust margin for the multi-tier legend
roi_plotter.fig.subplots_adjust(right=0.8)

# roi_plotter.show(axis_on=False, scale_nm=2, scale_linewidth=5, scale_fontsize=12)
roi_plotter.show(axis_on=False, scale_nm=1, scale_linewidth=5, scale_fontsize=12)

In [ ]:
roi.get_atom_positions()
higher_map_combined = higher_map_4 & higher_map_8 & higher_map_14

defect_dist = []
pristine_dist = []
for i in range(roi.grid.shape[0]):
    for j in range(roi.grid.shape[1]):
        if j > 0 and j < roi.grid.shape[1] - 1 and roi.grid[i, j].atom_type == 'Fe':  # Ensure we don't go out of bounds
            if higher_map_combined[i, j]:
                if roi.atom_positions[i, j+1, 1] - roi.atom_positions[i, j, 1] < roi.atom_positions[i, j, 1] - roi.atom_positions[i, j - 1, 1]:  # Only consider positive distances
                    defect_dist.append(roi.atom_positions[i, j+1, 1] - roi.atom_positions[i, j-1, 1])
                else:
                    defect_dist.append(roi.atom_positions[i, j+1, 1] - roi.atom_positions[i, j-1, 1])
            else:
                if roi.atom_positions[i, j+1, 1] - roi.atom_positions[i, j, 1] < roi.atom_positions[i, j, 1] - roi.atom_positions[i, j - 1, 1]:  # Only consider positive distances
                    pristine_dist.append(roi.atom_positions[i, j+1, 1] - roi.atom_positions[i, j-1, 1])
                else:
                    pristine_dist.append(roi.atom_positions[i, j+1, 1] - roi.atom_positions[i, j-1, 1])

defect_dist = np.array(defect_dist) * roi.scale * 10
pristine_dist = np.array(pristine_dist) * roi.scale * 10

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

"""
Plots a vertically stacked histogram sharing the x-axis with a gap between them.
The top subplot (defect) takes 1/5 of the height, and the bottom (pristine) takes 4/5.
"""

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'axes.labelsize': 18,     
    'xtick.labelsize': 14,    
    'ytick.labelsize': 14,
    'text.usetex': False 
})

color_pristine_hist = '#1f77b4'  
color_pristine_stat = "#001cbc"  
color_defect_hist = '#D6204E'  
color_defect_stat = "#9D0229"

# Calculate common bins so the x-axis aligns perfectly
min_val = min(np.min(defect_dist), np.min(pristine_dist))
max_val = max(np.max(defect_dist), np.max(pristine_dist))
custom_bins = np.linspace(min_val, max_val, 100)

fig, (ax_def, ax_norm) = plt.subplots(
    2, 1, 
    figsize=(6, 6), 
    sharex=True, 
    gridspec_kw={'height_ratios': [1, 4]}
)

def_mean = np.mean(defect_dist)
def_std = np.std(defect_dist)

ax_def.hist(defect_dist, bins=custom_bins, color=color_defect_hist, alpha=0.6, label='Detected Antisites')
ax_def.axvspan(xmin=def_mean - def_std, xmax=def_mean + def_std, color=color_defect_stat, alpha=0.15, label='Defect ±1 std.', linewidth=0)
ax_def.axvline(x=def_mean, color=color_defect_stat, linestyle='--', linewidth=1.5, label='Defect Mean')

trans_def = ax_def.get_xaxis_transform()
ax_def.text(def_mean, 0.95, f' {def_mean:.3f} Å', color=color_defect_stat, transform=trans_def, va='top', ha='left', fontsize=14, fontweight='bold')

ax_def.set_title('Lu-Lu Distances Across Fe Sites', fontsize=18, pad=15)
ax_def.tick_params(direction='in', top=False, right=False, length=6)

# Lock limits and enforce uniform tick spacing. 
tick_step = 9 
ax_def.set_ylim(0, 9)
# Since there is a gap now, we can safely start the ticks at 0 
ax_def.set_yticks(np.arange(0, 10, tick_step))

norm_mean = np.mean(pristine_dist)
norm_std = np.std(pristine_dist)

ax_norm.hist(pristine_dist, bins=custom_bins, color=color_pristine_hist, alpha=0.6, label='Detected Non-Antisites')
ax_norm.axvspan(xmin=norm_mean - norm_std, xmax=norm_mean + norm_std, color=color_pristine_stat, alpha=0.15, label='Pristine ±1 std.', linewidth=0)
ax_norm.axvline(x=norm_mean, color=color_pristine_stat, linestyle='--', linewidth=1.5, label='Pristine Mean')

trans_norm = ax_norm.get_xaxis_transform()
ax_norm.text(norm_mean, 0.95, f' {norm_mean:.3f} Å', color=color_pristine_stat, transform=trans_norm, va='top', ha='left', fontsize=14, fontweight='bold')

ax_norm.set_xlabel('Distance (Å)')
ax_norm.tick_params(direction='in', top=False, right=False, length=6)

# Match the tick_step from the top axis
ax_norm.set_ylim(0, 36)
ax_norm.set_yticks(np.arange(0, 37, tick_step))

# Center the 'Counts' label vertically across the entire figure
fig.supylabel('Counts', fontsize=18, x=0.00)

# Combine legends to the right of the plot
handles_def, labels_def = ax_def.get_legend_handles_labels()
handles_norm, labels_norm = ax_norm.get_legend_handles_labels()

ax_def.legend(handles_def, labels_def, loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=12)
ax_norm.legend(handles_norm, labels_norm, loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=12)

# Set hspace to 0.15 to create a nice visual gap between the subplots
fig.subplots_adjust(hspace=0.15, right=0.75, left=0.1, bottom=0.1) 
plt.show()

In [ ]:
import numpy as np

def estimate_defect_counts(map_4, map_8, map_14, sim_rates):
    """
    Estimates the absolute number of true defects in the experimental 
    image by dividing the raw detected counts by the simulation detection rates.
    """
    exp_4_count = np.sum(map_4)
    exp_8_count = np.sum(map_8)
    exp_14_count = np.sum(map_14)
    
    # Estimate the true number of defects using simulation efficiency
    est_4 = exp_4_count / sim_rates['4_atoms']
    est_8 = exp_8_count / sim_rates['8_atoms']
    est_14 = exp_14_count / sim_rates['14_atoms']
    
    print("Estimated True Defect Count (Calibrated to Simulation):")
    print(f"  Based on 4-atom metric:  {int(round(est_4))} defects (Raw detections: {exp_4_count})")
    print(f"  Based on 8-atom metric:  {int(round(est_8))} defects (Raw detections: {exp_8_count})")
    print(f"  Based on 14-atom metric: {int(round(est_14))} defects (Raw detections: {exp_14_count})")
    
    ensemble_avg = np.mean([est_4, est_8, est_14])
    print("-" * 55)
    print(f"  Ensemble Average Estimate: {int(round(ensemble_avg))} total defects")

simulated_detection_rates = {
    '4_atoms': 1,
    '8_atoms': 1,
    '14_atoms': 1
}

estimate_defect_counts(higher_map_4, higher_map_8, higher_map_14, simulated_detection_rates)

In [ ]:
import numpy as np

def count_valid_entries(data_array, mask):
    """
    Counts the number of elements that are both non-NaN in the data_array 
    and True in the provided boolean mask.
    """
    valid_data_mask = ~np.isnan(data_array)
    combined_mask = valid_data_mask & mask
    
    return np.sum(combined_mask)

valid_count = count_valid_entries(roi.relative_vicinity, fe_mask)
print(f"Number of valid entries: {valid_count}")

In [ ]:
roi_plotter = roi_tools.ROIPlotter(roi, title=None)
mean_intensity = roi.get_attribute_values(value_type='mean_intensity')

roi_plotter.add_patches_faces_overlay(atom_type = 'Fe', values = mean_intensity, cmap='viridis', alpha=0.05)
roi_plotter.add_boolean_edges_overlay(mask_sum == 3, color="#820021", linewidth=1)
roi_plotter.show(axis_on=False, scale_nm=1, scale_linewidth=5, scale_fontsize=12)